## Import libraries

In [1]:
import io
import sys
import random
import json

import numpy as np
import pandas as pd


#_________
from tqdm.auto import tqdm

## Utility functions

### safe_unzip()

In [2]:
import zipfile
import os

def safe_unzip(zip_path, extract_path=None):
    try:
        # Validate zip file exists
        if not os.path.exists(zip_path):
            raise FileNotFoundError(f"Zip file not found: {zip_path}")
        
        # Set extract path
        if extract_path is None:
            extract_path = os.path.splitext(zip_path)[0]
        
        # Create directory if needed
        os.makedirs(extract_path, exist_ok=True)
        
        # Extract
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            # Test if zip is valid
            bad_file = zip_ref.testzip()
            if bad_file:
                raise zipfile.BadZipFile(f"Corrupted file in zip: {bad_file}")
            
            zip_ref.extractall(extract_path)
            print(f"Successfully extracted to: {extract_path}")
            
    except FileNotFoundError as e:
        print(f"Error: {e}")
    except zipfile.BadZipFile as e:
        print(f"Error: Invalid or corrupted zip file - {e}")
    except Exception as e:
        print(f"Unexpected error: {e}")

### read_text_file()

In [3]:
def read_text_file(filepath):

    '''
    Description
    -----------

        =================
        Phonetic info
        this is the structure of a .phn file 
            0 3050 h#
            3050 4559 sh
            4559 5723 ix
            5723 6642 hv
            6642 8772 eh
            8772 9190 dcl
            9190 10337 jh

            as you can see it is seperator by space, it looks like time points (for sr=16000)
                and the last [-1] column is the one with the Arphable phonetic symbole. 


        =================
        word info
        similar happens with .wrd file 
            3050 5723 she
            5723 10337 had
            9190 11517 your
            11517 16334 dark
            16334 21199 suit
            21199 22560 in
            22560 28064 greasy
            28064 33360 wash
            33754 37556 water
            37556 40313 all
            40313 44586 year
        as you can see it is seperator by space, it looks like time points (for sr=16000)
            and the last [-1] column per line is the one that contains the word. 
    
    '''

    
    with open(filepath) as f:
        tokens = [line.split()[-1] for line in f]
        return " ".join(tokens)

## Unzip

In [4]:
TIMIT_KAGGLE_ROOT = os.path.join("..", "..", "..",
                                 'data', 
                                 '07_timit_kaggle'
                                 
                                )
#=========================================================
data_path = os.path.join(TIMIT_KAGGLE_ROOT, 'data') 

In [5]:
%%time
zip_fname = 'timit_kaggle.zip'
zip_fpath = os.path.join(TIMIT_KAGGLE_ROOT, zip_fname)

zip_extract_path = TIMIT_KAGGLE_ROOT

safe_unzip(zip_path=zip_fpath, extract_path=zip_extract_path)
print(f"zip file has been extrated in this folder: \n {zip_extract_path}")

Error: Zip file not found: ../../../data/07_timit_kaggle/timit_kaggle.zip
zip file has been extrated in this folder: 
 ../../../data/07_timit_kaggle
CPU times: user 689 μs, sys: 0 ns, total: 689 μs
Wall time: 12.9 ms


## Load meta timit

In [6]:
df_train = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'train_data.csv'))
df_train.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False


In [7]:
df_test = pd.read_csv(os.path.join(TIMIT_KAGGLE_ROOT, 'test_data.csv'))
df_test.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TEST,DR4,MGMM0,SX139.WAV,TEST/DR4/MGMM0/SX139.WAV,TEST\\DR4\\MGMM0\\SX139.WAV,False,True,False,False,False
1,2.0,TEST,DR4,MGMM0,SX139.WAV.wav,TEST/DR4/MGMM0/SX139.WAV.wav,TEST\\DR4\\MGMM0\\SX139.WAV.wav,True,True,False,False,False
2,3.0,TEST,DR4,MGMM0,SX139.TXT,TEST/DR4/MGMM0/SX139.TXT,TEST\\DR4\\MGMM0\\SX139.TXT,False,False,False,False,True


In [8]:
df = pd.concat([df_train, df_test])
df.head(3)

,index,test_or_train,dialect_region,speaker_id,filename,path_from_data_dir,path_from_data_dir_windows,is_converted_audio,is_audio,is_word_file,is_phonetic_file,is_sentence_file
0,1.0,TRAIN,DR4,MMDM0,SI681.WAV.wav,TRAIN/DR4/MMDM0/SI681.WAV.wav,TRAIN\\DR4\\MMDM0\\SI681.WAV.wav,True,True,False,False,False
1,2.0,TRAIN,DR4,MMDM0,SI1311.PHN,TRAIN/DR4/MMDM0/SI1311.PHN,TRAIN\\DR4\\MMDM0\\SI1311.PHN,False,False,False,True,False
2,3.0,TRAIN,DR4,MMDM0,SI1311.WRD,TRAIN/DR4/MMDM0/SI1311.WRD,TRAIN\\DR4\\MMDM0\\SI1311.WRD,False,False,True,False,False


In [9]:
c1 = df['is_converted_audio'] == False
df = df[c1].reset_index(drop=True)
df.shape

(25200, 12)

In [10]:
%%time
data = {}

for idx, row in tqdm(df.iterrows()):

    # meta file with linux sys path sep to audio file. 
    #  Each audio file name is related to subID
    #  the root folder that a "data" folder with all wav file, phonetic, word files. 
    path = row['path_from_data_dir'] 

    # Extracting from path of wav the subId path. 
    entry_id = path.split('.')[0]

    #__________________________________
    # Creating empty dictionary to story subID info.
    if entry_id not in data:
        data[entry_id] = {}

    #__________________________________
    # data_path is the root in the system to "data" folder in TIMIT kaggle root folder.
    if row['is_audio'] is True:
        data[entry_id]['audio_file'] = os.path.join(data_path, path)

    elif row['is_word_file'] is True:
        data[entry_id]['word_file'] = os.path.join(data_path, path)

    elif row['is_phonetic_file'] is True:
        data[entry_id]['phonetic_file'] = os.path.join(data_path, path)


    #__________________________________
    
    # break

0it [00:00, ?it/s]

CPU times: user 1.31 s, sys: 15.2 ms, total: 1.33 s
Wall time: 1.45 s


In [11]:
len(data.keys()) # 

6300

### Extracting subID with 3 files

In [12]:
keys = [key for key in data.keys() if len(data[key]) == 3]
len(keys)

3360

### Split data into train,val,test

In [13]:
random.Random(101).shuffle(keys)

num_train = int(len(keys) * 0.8)
num_valid = int(len(keys) * 0.1)
num_test = len(keys) - num_train - num_valid

train_keys = keys[:num_train]
valid_keys = keys[num_train:num_train + num_valid]
test_keys = keys[-num_test:]

In [14]:
print(f"train_keys {len(train_keys )}")
print(f"valid_keys {len(valid_keys )}")
print(f"test_keys {len(test_keys )}")

train_keys 2688
valid_keys 336
test_keys 336


In [15]:
# form the data dictionary and with the keys split into train, val and test, organize data for each set 
train = { key:data[key] for key in train_keys }
valid = { key:data[key] for key in valid_keys }
test  = { key:data[key] for key in test_keys }

### SAVE the dictionary as json files 

In [16]:
custom_split_data_path = os.path.join(TIMIT_KAGGLE_ROOT, "01_custom_split_meta")

os.makedirs(custom_split_data_path, exist_ok=True)
custom_split_data_path

'../../../data/07_timit_kaggle/01_custom_split_meta'

In [17]:
fnames = ['custom_train.json', 'custom_valid.json', 'custom_test.json']

for fname in fnames: 
    fpath = os.path.join(custom_split_data_path,fname )
    with open(fpath, "w") as f:
        json.dump(train, f)

## Extracting one file .phn

In [18]:
sub_ids_lst = list(train.keys())
sub_ids_lst[:5]

['TEST/DR7/FISB0/SA1',
 'TRAIN/DR4/MAEB0/SI1411',
 'TRAIN/DR2/FCAJ0/SX39',
 'TRAIN/DR3/MILB0/SX93',
 'TEST/DR2/MMDM2/SI2082']

In [19]:
sID = sub_ids_lst[0]
sID

'TEST/DR7/FISB0/SA1'

In [20]:
train[sID]

{'audio_file': '../../../data/07_timit_kaggle/data/TEST/DR7/FISB0/SA1.WAV',
 'phonetic_file': '../../../data/07_timit_kaggle/data/TEST/DR7/FISB0/SA1.PHN',
 'word_file': '../../../data/07_timit_kaggle/data/TEST/DR7/FISB0/SA1.WRD'}

In [21]:
# phones_seg = read_text_file(filepath=)
# phones_seg

In [22]:
fpath_phone = train[sID]['phonetic_file']
filepath = fpath_phone
with open(filepath) as f:
    tokens = [line.split() for line in f]
d_phone = pd.DataFrame(tokens, columns=["start", "end", "label"])
d_phone['start'] = d_phone['start'].astype(int)
d_phone['end'] = d_phone['end'].astype(int)
d_phone.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41 entries, 0 to 40
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   start   41 non-null     int64 
 1   end     41 non-null     int64 
 2   label   41 non-null     object
dtypes: int64(2), object(1)
memory usage: 1.1+ KB


## Creating dummy pred .phn file 

In [23]:
## Creating dummy prediction
sr = 16000
range_width = sr*.025 # 25ms

d_phone_pred = d_phone.copy()

d_phone_pred["start_pred"] = d_phone_pred["start"].apply(
    lambda x: np.random.randint(x - range_width, x + range_width + 1)
)

d_phone_pred["end_pred"] = d_phone_pred["end"].apply(
    lambda x: np.random.randint(x - range_width, x + range_width + 1)
)

cols = ["start_pred", "end_pred", "label"]
d_phone_pred = d_phone_pred[cols].copy()
d_phone_pred

,start_pred,end_pred,label
0,196,11651,h#
1,11123,13410,sh
2,13434,14891,iy
3,14783,16918,hv
4,16550,18217,ae
5,18610,19121,dcl
6,19294,19560,d
7,19321,20938,y
8,20689,23318,er
9,23218,24679,dcl


## Sequence matcher (two list of strings)

In [24]:
import difflib

In [48]:
ground_truth = ['w', 'O', 'r', 'd', 's']
asr_output = ['w', 'o', 'r', 'd','s','z']

In [49]:
## class instantiation
matcher = difflib.SequenceMatcher(isjunk=None, 
                                  a=ground_truth, 
                                  b=asr_output,
                                 )

In [31]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'equal':
        for i in range(i2 - i1):
            print(f"✅ Correct: {ground_truth[i1 + i]}")
    elif tag == 'replace':
        for i in range(max(i2 - i1, j2 - j1)):
            g = ground_truth[i1 + i] if i1 + i < i2 else "-"
            a = asr_output[j1 + i] if j1 + i < j2 else "-"
            print(f"❌ Wrong: {g} → {a}")
    elif tag == 'delete':
        for i in range(i2 - i1):
            prev = ground_truth[i1 + i - 1] if i1 + i - 1 >= 0 else None
            nxt = ground_truth[i1 + i + 1] if i1 + i + 1 < len(ground_truth) else None
            print(f"🕳️ Missing: {ground_truth[i1 + i]} (between {prev} and {nxt})")
    elif tag == 'insert':
        for i in range(j2 - j1):
            print(f"➕ Extra: {asr_output[j1 + i]} (not in Expected Sound)")

✅ Correct: w
❌ Wrong: O → o
✅ Correct: r
✅ Correct: d
✅ Correct: s
➕ Extra: z (not in Expected Sound)


In [51]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes(): 
    print(tag, i1, i2, j1, j2)

equal 0 1 0 1
replace 1 2 1 2
equal 2 5 2 5
insert 5 5 5 6


## List of phonemes match

In [61]:
phone_pred = d_phone_pred["label"].to_list()
phone_truth= d_phone["label"].to_list()

In [62]:
## class instantiation
matcher = difflib.SequenceMatcher(isjunk=None, 
                                  a=phone_truth, 
                                  b=phone_pred,
                                 )

In [63]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'equal':
        for i in range(i2 - i1):
            print(f"✅ Correct: {phone_truth[i1 + i]}")
    elif tag == 'replace':
        for i in range(max(i2 - i1, j2 - j1)):
            g = phone_truth[i1 + i] if i1 + i < i2 else "-"
            a = phone_pred[j1 + i] if j1 + i < j2 else "-"
            print(f"❌ Wrong: {g} → {a}")
    elif tag == 'delete':
        for i in range(i2 - i1):
            prev = phone_truth[i1 + i - 1] if i1 + i - 1 >= 0 else None
            nxt = phone_truth[i1 + i + 1] if i1 + i + 1 < len(phone_truth) else None
            print(f"🕳️ Missing: {phone_truth[i1 + i]} (between {prev} and {nxt})")
    elif tag == 'insert':
        for i in range(j2 - j1):
            print(f"➕ Extra: {phone_pred[j1 + i]} (not in Expected Sound)")

✅ Correct: h#
✅ Correct: sh
✅ Correct: iy
✅ Correct: hv
✅ Correct: ae
✅ Correct: dcl
✅ Correct: d
✅ Correct: y
✅ Correct: er
✅ Correct: dcl
✅ Correct: d
✅ Correct: aa
✅ Correct: r
✅ Correct: kcl
✅ Correct: k
✅ Correct: s
✅ Correct: ux
✅ Correct: tcl
✅ Correct: ih
✅ Correct: n
✅ Correct: gcl
✅ Correct: g
✅ Correct: r
✅ Correct: iy
✅ Correct: z
✅ Correct: iy
✅ Correct: w
✅ Correct: ao
✅ Correct: sh
✅ Correct: epi
✅ Correct: w
✅ Correct: ao
✅ Correct: dx
✅ Correct: er
✅ Correct: q
✅ Correct: ao
✅ Correct: l
✅ Correct: y
✅ Correct: ih
✅ Correct: axr
✅ Correct: h#


In [64]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes(): 
    print(tag, i1, i2, j1, j2)

equal 0 41 0 41


In [65]:
print(phone_truth)

['h#', 'sh', 'iy', 'hv', 'ae', 'dcl', 'd', 'y', 'er', 'dcl', 'd', 'aa', 'r', 'kcl', 'k', 's', 'ux', 'tcl', 'ih', 'n', 'gcl', 'g', 'r', 'iy', 'z', 'iy', 'w', 'ao', 'sh', 'epi', 'w', 'ao', 'dx', 'er', 'q', 'ao', 'l', 'y', 'ih', 'axr', 'h#']


In [66]:
print(phone_truth[i1:i2])

['h#', 'sh', 'iy', 'hv', 'ae', 'dcl', 'd', 'y', 'er', 'dcl', 'd', 'aa', 'r', 'kcl', 'k', 's', 'ux', 'tcl', 'ih', 'n', 'gcl', 'g', 'r', 'iy', 'z', 'iy', 'w', 'ao', 'sh', 'epi', 'w', 'ao', 'dx', 'er', 'q', 'ao', 'l', 'y', 'ih', 'axr', 'h#']


## testing scenarios

**when calculating Phone recognition metrics in the correct time period** several scenarios can occur. 
Lets account only for phoneme sequence prediction
- It could be correct.
- It could replace the ground truth token for other one
- it could insert a new ground truth token.
- it could delete (omit) the prediction of a ground truth token

Handling this scenarios is relevant if you want to create a metric that consider both task 
- Phoneme recognition and
- Phoneme segmentation (correct phoneme in correct time segment)

### modification

In [74]:
## Insert
rows = d_phone_pred.values.tolist()
rows.insert(5, [18610, 19121, "ff"])
d_phone_pred1 = pd.DataFrame(rows, columns=d_phone_pred.columns)
d_phone_pred1

,start_pred,end_pred,label
0,196,11651,h#
1,11123,13410,sh
2,13434,14891,iy
3,14783,16918,hv
4,16550,18217,ae
5,18610,19121,ff
6,18610,19121,dcl
7,19294,19560,d
8,19321,20938,y
9,20689,23318,er


In [79]:
a = [1, 2, 3,]
a.pop(1)
a

[1, 3]

In [80]:
## remove 
rows = d_phone_pred1.values.tolist()
rows.pop(10)
d_phone_pred2 = pd.DataFrame(rows, columns=d_phone_pred.columns)
d_phone_pred2

,start_pred,end_pred,label
0,196,11651,h#
1,11123,13410,sh
2,13434,14891,iy
3,14783,16918,hv
4,16550,18217,ae
5,18610,19121,ff
6,18610,19121,dcl
7,19294,19560,d
8,19321,20938,y
9,20689,23318,er


In [81]:
## replace
rows = d_phone_pred2.values.tolist()
rows.pop(20)
rows.insert(20, [41717, 41827, "ii"])
d_phone_pred3 = pd.DataFrame(rows, columns=d_phone_pred.columns)
d_phone_pred3

,start_pred,end_pred,label
0,196,11651,h#
1,11123,13410,sh
2,13434,14891,iy
3,14783,16918,hv
4,16550,18217,ae
5,18610,19121,ff
6,18610,19121,dcl
7,19294,19560,d
8,19321,20938,y
9,20689,23318,er


## mapping scenarios to ground true sequence and time

In [82]:
phone_pred = d_phone_pred3["label"].to_list()
phone_truth= d_phone["label"].to_list()

In [83]:
## class instantiation
matcher = difflib.SequenceMatcher(isjunk=None, 
                                  a=phone_truth, 
                                  b=phone_pred,
                                 )

In [84]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes():
    if tag == 'equal':
        for i in range(i2 - i1):
            print(f"✅ Correct: {phone_truth[i1 + i]}")
    elif tag == 'replace':
        for i in range(max(i2 - i1, j2 - j1)):
            g = phone_truth[i1 + i] if i1 + i < i2 else "-"
            a = phone_pred[j1 + i] if j1 + i < j2 else "-"
            print(f"❌ Wrong: {g} → {a}")
    elif tag == 'delete':
        for i in range(i2 - i1):
            prev = phone_truth[i1 + i - 1] if i1 + i - 1 >= 0 else None
            nxt = phone_truth[i1 + i + 1] if i1 + i + 1 < len(phone_truth) else None
            print(f"🕳️ Missing: {phone_truth[i1 + i]} (between {prev} and {nxt})")
    elif tag == 'insert':
        for i in range(j2 - j1):
            print(f"➕ Extra: {phone_pred[j1 + i]} (not in Expected Sound)")

✅ Correct: h#
✅ Correct: sh
✅ Correct: iy
✅ Correct: hv
✅ Correct: ae
➕ Extra: ff (not in Expected Sound)
✅ Correct: dcl
✅ Correct: d
✅ Correct: y
✅ Correct: er
🕳️ Missing: dcl (between er and d)
✅ Correct: d
✅ Correct: aa
✅ Correct: r
✅ Correct: kcl
✅ Correct: k
✅ Correct: s
✅ Correct: ux
✅ Correct: tcl
✅ Correct: ih
✅ Correct: n
❌ Wrong: gcl → ii
✅ Correct: g
✅ Correct: r
✅ Correct: iy
✅ Correct: z
✅ Correct: iy
✅ Correct: w
✅ Correct: ao
✅ Correct: sh
✅ Correct: epi
✅ Correct: w
✅ Correct: ao
✅ Correct: dx
✅ Correct: er
✅ Correct: q
✅ Correct: ao
✅ Correct: l
✅ Correct: y
✅ Correct: ih
✅ Correct: axr
✅ Correct: h#


In [85]:
for tag, i1, i2, j1, j2 in matcher.get_opcodes(): 
    print(tag, i1, i2, j1, j2)

equal 0 5 0 5
insert 5 5 5 6
equal 5 9 6 10
delete 9 10 10 10
equal 10 20 10 20
replace 20 21 20 21
equal 21 41 21 41


In [198]:
#_____________________________________________
# Transforming this to list for better manipulation
rows_truth = d_phone.values.tolist()
rows_pred = d_phone_pred3.values.tolist()
#_____________________________________________

tag_lst = []
#_____________________________________________
for idx, (tag, i1, i2, j1, j2) in enumerate(matcher.get_opcodes()):
    # print(idx, tag, i1, i2, j1, j2)

    #___________________________________________
    # Pred label equal
    
    if tag == 'equal':

        lst1 = rows_truth[i1:i2]
        lst2 = rows_pred[j1:j2]

        tag = [[i] for i in ["correct"]*len(lst1)]
        
        correct_lst = [a + b for a, b in zip(lst1, lst2)]
        correct_lst = [a + b for a, b in zip(correct_lst, tag)]
        
        tag_lst.append(correct_lst)
        
        del correct_lst, lst1, lst2, tag
    #___________________________________________
    # Insert scenario
    elif tag == 'insert':

        lst1 = rows_truth[i1:i2]
        lst2 = rows_pred[j1:j2]
        
        #------- 
        # if inserted, then for each col in ground true I insert nan
        if len(lst1) == 0: 
            lst1 = [[np.nan, np.nan, np.nan]]*len(lst2)
        
        tag = [[i] for i in ["insert"]*len(lst2)]

        insert_lst = [a + b for a, b in zip(lst1, lst2)]
        insert_lst = [a + b for a, b in zip(insert_lst, tag)]

        tag_lst.append(insert_lst)
        
        del insert_lst, lst1, lst2, tag

    #___________________________________________
    # if prediction didnt predict one element
    elif tag == 'delete':


        lst1 = rows_truth[i1:i2]
        lst2 = rows_pred[j1:j2]

        if len(lst2) == 0:
            lst2 = [[np.nan, np.nan, np.nan]]*len(lst1)

        tag = [[i] for i in ["missing"]*len(lst2)]
        
        miss_lst = [a + b for a, b in zip(lst1, lst2)]
        miss_lst = [a + b for a, b in zip(miss_lst, tag)]

        tag_lst.append(miss_lst)

        del miss_lst, lst1, lst2, tag
            
    #___________________________________________
    # If the prediction replace the sequence. 
    elif tag == 'replace':

        lst1 = rows_truth[i1:i2]
        lst2 = rows_pred[j1:j2]
        
        tag = [[i] for i in ["replace"]*len(lst2)]

        replace_lst = [a + b for a, b in zip(lst1, lst2)]
        replace_lst = [a + b for a, b in zip(replace_lst, tag)]

        tag_lst.append(replace_lst)

        del replace_lst, lst1, lst2, tag
        
    #________________________________________
    # if idx == 5: 
    #     print(idx)
    #     break
#____________________________________________
#============================================
# tag is is a list of 2D lists, a 3D list, concact to get 2D list
l0 = tag_lst[0]
for l in tag_lst:
    l0 = l0+l
#_________________________
# to pd.DataFrame for easier querying.
cols = ["start", "end", "label_true", "start_p", "end_p", "label_p", "align_cat"]
d_grnd_pred = pd.DataFrame(l0, columns=cols)
d_grnd_pred

,start,end,label_true,start_p,end_p,label_p,align_cat
0,0.0,11289.0,h#,196.0,11651.0,h#,correct
1,11289.0,13323.0,sh,11123.0,13410.0,sh,correct
2,13323.0,14640.0,iy,13434.0,14891.0,iy,correct
3,14640.0,16751.0,hv,14783.0,16918.0,hv,correct
4,16751.0,18374.0,ae,16550.0,18217.0,ae,correct
5,0.0,11289.0,h#,196.0,11651.0,h#,correct
6,11289.0,13323.0,sh,11123.0,13410.0,sh,correct
7,13323.0,14640.0,iy,13434.0,14891.0,iy,correct
8,14640.0,16751.0,hv,14783.0,16918.0,hv,correct
9,16751.0,18374.0,ae,16550.0,18217.0,ae,correct


In [201]:
c1 = d_grnd_pred['align_cat'] != "correct" 
d_grnd_pred[c1].shape[0]/d_grnd_pred.shape[0]

0.06382978723404255

## Distance between boundary and pred_bound

In [ ]:
d_grnd_pred